# Data Cleaning & Feature Engineering

## In this section:
- We clean the dataset (handle missing values, duplicates, incorrect entries)
- We create new features (feature engineering)
- We prepare the dataset for EDA and ML 

Importing Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
print("Libraries imported successfully!")

Libraries imported successfully!


##  Loading the Dataset
In this step, we load the preprocessed dataset into a Pandas DataFrame for cleaning and feature engineering.

In [2]:
df_hotel=pd.read_csv('../datasets/hotel_final.csv')
df_hotel.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,reservation_status_date,city,arrival_date,lat,lon,temperature_2m_mean,rain_sum,snowfall_sum,sunrise,sunset
0,Resort Hotel - Chandigarh,0,342,2024,July,30,27,0,0,2,...,2024-07-27 22:16:40.916332324,Chandigarh,2024-07-27,30.733442,76.779714,31.449999,3.300000,0.0,1722038852,1722088254
1,Resort Hotel - Mumbai,0,737,2024,April,17,28,0,0,2,...,2024-04-28 21:56:21.507509066,Mumbai,2024-04-28,19.054999,72.869203,33.218746,0.000000,0.0,1714264928,1714311000
2,Resort Hotel - Delhi,0,7,2024,September,37,10,0,1,1,...,2024-09-10 03:46:25.734029096,Delhi,2024-09-10,28.613895,77.209006,27.170832,9.600000,0.0,1725928440,1725973337
3,Resort Hotel - Kolkata,0,13,2024,August,33,14,0,1,1,...,2024-08-14 18:07:10.049669568,Kolkata,2024-08-14,22.572646,88.363895,28.131250,39.999996,0.0,1723592606,1723639153
4,Resort Hotel - Lucknow,0,14,2024,September,37,14,0,2,2,...,2024-09-14 14:27:32.473846000,Lucknow,2024-09-14,26.838100,80.934600,26.945833,0.100000,0.0,1726273292,1726317720


In [3]:
df_hotel["sunrise_dt"] = pd.to_datetime(df_hotel["sunrise"], unit="s", utc=True)
df_hotel["sunset_dt"] = pd.to_datetime(df_hotel["sunset"], unit="s", utc=True)
df_hotel['sunrise_dt']=df_hotel['sunrise_dt'].astype(str)
df_hotel['sunrise_dt']=df_hotel['sunrise_dt'].str[11:-12]
df_hotel['sunrise_dt'].value_counts()

sunrise_dt
00    70959
01    39735
23     8696
Name: count, dtype: int64

These hours are very close to each other, and there is no significant difference between them, so they are being dropped.

In [4]:
df_hotel=df_hotel.drop(columns=['lon','lat','sunrise','sunset','sunset_dt','sunrise_dt'])

In [5]:
df_hotel.sample(5)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,city,arrival_date,temperature_2m_mean,rain_sum,snowfall_sum
50652,City Hotel - Bhopal,1,259,2024,April,16,17,0,3,2,...,90.00,0,0,Canceled,2024-04-17 12:15:20.888859108,Bhopal,2024-04-17,30.781252,0.0,0.0
107633,City Hotel - Goa,0,22,2024,September,36,2,1,2,2,...,121.33,0,0,Check-Out,2024-09-02 06:19:23.504175424,Goa,2024-09-02,24.741669,11.2,0.0
2772,Resort Hotel - Chandigarh,1,155,2024,December,52,24,0,1,2,...,36.00,0,0,Canceled,2024-12-24 22:52:01.312683748,Chandigarh,2024-12-24,12.429168,0.0,0.0
84396,City Hotel - Bangalore,0,5,2024,August,33,18,0,1,1,...,71.84,0,1,Check-Out,2024-08-18 15:40:14.845588788,Bangalore,2024-08-18,24.383337,3.1,0.0
35067,Resort Hotel - Delhi,0,63,2024,December,49,7,1,2,1,...,63.95,0,0,Check-Out,2024-12-07 16:27:10.702996088,Delhi,2024-12-07,16.556250,0.0,0.0


In [6]:
first_shape_of_df=df_hotel.shape
first_shape_of_df

(119390, 37)

Our dataset contains 119,390 rows and 37 columns.

In [7]:
df_hotel.duplicated().sum()

np.int64(0)

There are no duplicate rows in the dataset.

In [8]:
df_summary = pd.DataFrame({
    'dtype': df_hotel.dtypes,
    'missing_values': df_hotel.isnull().mean()*100
})
df_summary


,dtype,missing_values
hotel,object,0.000000
is_canceled,int64,0.000000
lead_time,int64,0.000000
arrival_date_year,int64,0.000000
arrival_date_month,object,0.000000
arrival_date_week_number,int64,0.000000
arrival_date_day_of_month,int64,0.000000
stays_in_weekend_nights,int64,0.000000
stays_in_week_nights,int64,0.000000
adults,int64,0.000000


Some columns have incorrect data types, and some contain null values.

In [9]:
cols_to_drop=[]
for col,value in (df_hotel.isna().mean()*100).items():
    if value>0 and value<=5:
        cols_to_drop.append(col)
        print(f'column: {col}, Mean percentage of null values: {value}')

column: children, Mean percentage of null values: 0.003350364352123293
column: country, Mean percentage of null values: 0.40874445095904177


We select columns where the percentage of null values is less than 5% and create a list of these columns for potential removal.

In [10]:
df_hotel=df_hotel.dropna(subset=cols_to_drop)
df_hotel=df_hotel.drop(columns='company')

At the same time, we are dropping the 'company' column since it has 94% null values.

In [11]:
df_hotel['agent']=df_hotel['agent'].fillna(0)
df_hotel['agent']=df_hotel['agent'].astype(str)

For the 'agent' column, we fill null values with 0, as these may not represent true missing data but simply indicate that there is no agent for that row. Since this column represents an ID, we also convert it to a string type.


In [12]:
df_hotel[['reservation_status_date','reservation_status','is_canceled']].sample(10)

,reservation_status_date,reservation_status,is_canceled
65581,2024-11-05 09:24:27.723157072,Canceled,1
20086,2024-05-23 07:54:02.982184288,Check-Out,0
102762,2024-10-20 21:59:26.047123268,Check-Out,0
33593,2024-10-07 14:09:18.262486492,Check-Out,0
55192,2024-08-09 09:47:20.553149788,Canceled,1
17369,2024-03-27 19:53:00.383452412,Check-Out,0
115649,2024-06-17 14:02:24.314802870,Check-Out,0
83825,2024-10-25 00:45:05.133638776,Check-Out,0
28641,2024-01-25 17:04:43.533658879,Check-Out,0
97342,2024-10-20 12:13:54.770372480,Check-Out,0


In [13]:
df_hotel=df_hotel.drop(columns=['reservation_status_date','reservation_status'])

The reservation_status column already conveys the same information as is_canceled, so we selected it along with other unnecessary columns and removed them from the dataset.

In [14]:
for c in df_hotel.columns:
    print(c,df_hotel[c].nunique())

hotel 30
is_canceled 2
lead_time 479
arrival_date_year 1
arrival_date_month 12
arrival_date_week_number 52
arrival_date_day_of_month 31
stays_in_weekend_nights 15
stays_in_week_nights 33
adults 14
children 5
babies 5
meal 5
country 177
market_segment 7
distribution_channel 5
is_repeated_guest 2
previous_cancellations 15
previous_bookings_not_canceled 73
reserved_room_type 10
assigned_room_type 12
booking_changes 21
deposit_type 3
agent 333
days_in_waiting_list 128
customer_type 4
adr 8870
required_car_parking_spaces 5
total_of_special_requests 6
city 15
arrival_date 366
temperature_2m_mean 4793
rain_sum 823
snowfall_sum 1


We check the number of unique values in each column. The 'arrival_date_year' and 'snowfall_sum' columns have only one unique value, and the 'arrival_date_week' column indicates the week number of the year, so these columns will be dropped.


In [15]:
df_hotel=df_hotel.drop(columns=['arrival_date_year','snowfall_sum','arrival_date_week_number','arrival_date'])

In [16]:
df_hotel['distribution_channel'].value_counts()

distribution_channel
TA/TO        97730
Direct       14483
Corporate     6491
GDS            193
Undefined        1
Name: count, dtype: int64

Since the number of undefined values is only 1 and it accounts for less than 5% of the new dataset, we remove it.

In [17]:
df_hotel=df_hotel[df_hotel['distribution_channel']!='Undefined']

Then, we analyze the categorical variables by inspecting their unique value counts and the number of observations in each category.

In [18]:
cat_columns=['hotel','is_canceled','arrival_date_month','meal','country','market_segment','distribution_channel','deposit_type','agent','customer_type','city']
for c in cat_columns:
    print(df_hotel[c].nunique())
    print(df_hotel[c].value_counts())

30
hotel
City Hotel - Ahmedabad       5403
City Hotel - Bhopal          5367
City Hotel - Jaipur          5342
City Hotel - Pune            5339
City Hotel - Kolkata         5337
City Hotel - Hyderabad       5336
City Hotel - Goa             5296
City Hotel - Chandigarh      5291
City Hotel - Lucknow         5289
City Hotel - Mumbai          5260
City Hotel - Chennai         5251
City Hotel - Delhi           5249
City Hotel - Bangalore       5208
City Hotel - Indore          5169
City Hotel - Kochi           5165
Resort Hotel - Bhopal        2703
Resort Hotel - Delhi         2693
Resort Hotel - Kochi         2689
Resort Hotel - Jaipur        2668
Resort Hotel - Bangalore     2662
Resort Hotel - Chandigarh    2657
Resort Hotel - Mumbai        2650
Resort Hotel - Goa           2647
Resort Hotel - Chennai       2646
Resort Hotel - Indore        2627
Resort Hotel - Pune          2608
Resort Hotel - Hyderabad     2604
Resort Hotel - Kolkata       2600
Resort Hotel - Ahmedabad     2591
Resor

Some columns contain a limited number of unique categorical values, whereas others exhibit a high level of categorical diversity.

In [19]:
df_hotel[['children','adults','babies']].sample(5)

,children,adults,babies
2455,0.0,2,0
66498,0.0,2,0
61772,0.0,2,0
111418,0.0,1,0
42350,0.0,2,0


In [20]:
df_hotel['people']=df_hotel['children']+df_hotel['adults']+df_hotel['babies']
df_hotel=df_hotel.drop(columns=['children','adults','babies'])
result=((df_hotel[df_hotel['people']==0]).shape[0])/(first_shape_of_df[0])*100
if result<=5 and result>0:
    print(f'Since the result value--->({np.round(result,3)}%) does not exceed 5%, those observations were removed from the dataset.')
    df_hotel=df_hotel[df_hotel['people']!=0]
else:
    print('As a result, no values were removed from the dataset.')

Since the result value--->(0.142%) does not exceed 5%, those observations were removed from the dataset.


Next, we examine the adults, babies, and children columns. Instead of keeping them separately, we create a new feature called people, representing the total number of guests. We also remove observations where the total number of people is 0, as this is logically inconsistent. However, before doing so, we first check whether these cases constitute less than 5% of the dataset, since removing a larger portion could be risky.

In [21]:
df_hotel[['reserved_room_type','assigned_room_type']].sample(10)

,reserved_room_type,assigned_room_type
62709,A,A
79151,A,A
8852,E,E
105451,A,A
74214,A,A
96435,E,E
81188,A,D
93411,D,D
79880,A,A
18110,A,D


In [22]:
df_hotel['same_room']=df_hotel.apply(lambda x: 'Yes' if  x['reserved_room_type']==x['assigned_room_type'] else 'No',axis=1)
df_hotel=df_hotel.drop(columns=['reserved_room_type','assigned_room_type'])

We create a new feature called same_room based on the assigned_room and reserved_room columns. If the assigned room is the same as the reserved room, same_room is marked as "Yes"; otherwise, it is marked as "No". The original assigned_room and reserved_room columns are then dropped to avoid redundancy.

In [23]:
df_hotel['total_stays']=df_hotel['stays_in_weekend_nights']+df_hotel['stays_in_week_nights']
df_hotel=df_hotel.drop(columns=['stays_in_weekend_nights','stays_in_week_nights'])

We created a new total_stays column by combining stays_in_weekend_nights and stays_in_week_nights. The original two columns were then dropped to avoid redundancy.

In [24]:
df_hotel[['city','hotel']].sample(10)

,city,hotel
92277,Delhi,City Hotel - Delhi
8795,Chandigarh,Resort Hotel - Chandigarh
77901,Mumbai,City Hotel - Mumbai
75540,Pune,City Hotel - Pune
22666,Kolkata,Resort Hotel - Kolkata
13450,Bhopal,Resort Hotel - Bhopal
15661,Mumbai,Resort Hotel - Mumbai
17690,Pune,Resort Hotel - Pune
84947,Bangalore,City Hotel - Bangalore
9553,Bangalore,Resort Hotel - Bangalore


In [25]:
df_hotel['checking']=df_hotel.apply(lambda x:'Correct' if x['city'] in x['hotel'] else 'Wrong',axis=1)
result=df_hotel['checking'].unique()
df_hotel=df_hotel.drop(columns='checking')
result[0]

'Correct'

The hotel column shows the hotel names along with their respective cities. The city column separately indicates the city where each hotel is located. We checked whether there are any inconsistencies in any row — whether hotel names and city names are correctly aligned. As a result, we found that all entries are correct, and there are no issues.

In [26]:
condition = (((df_hotel['previous_bookings_not_canceled'] > 0) |(df_hotel['previous_cancellations'] > 0)) & (df_hotel['is_repeated_guest'] == 0))
if condition.any():
    df_hotel.loc[condition, 'is_repeated_guest'] = 1
    print('The replace operation was executed.')
else:
    print('No replacements occurred.')

The replace operation was executed.


The previous_cancellations column shows the number of prior bookings that were canceled, while previous_bookings_not_canceled shows the number of prior bookings that were not canceled. The is_repeated_guest column has values 0 and 1, where 1 indicates that the guest has visited before. If either previous_cancellations or previous_bookings_not_canceled is greater than 0, it implies that the guest is a repeat visitor. If is_repeated_guest shows 0 in such cases, it is logically inconsistent. Therefore, we corrected some of these values by setting them to 1.

In [27]:
def other(data,columns):
    for col in columns:
        val_count=data[col].value_counts()
        if (val_count<30).any():
            print(f"This column->({col}) contains rare categories with fewer than 30 occurrences.")
            lower_than_30=val_count[val_count<30]
            if lower_than_30.sum()>=30:
                print("Rare categories in this column (count < 30) were grouped into an 'Other' category, as their combined frequency is at least 30.")
                data[col]=data[col].replace(lower_than_30.index,'Other')

In [28]:
other(df_hotel,cat_columns)

This column->(country) contains rare categories with fewer than 30 occurrences.
Rare categories in this column (count < 30) were grouped into an 'Other' category, as their combined frequency is at least 30.
This column->(agent) contains rare categories with fewer than 30 occurrences.
Rare categories in this column (count < 30) were grouped into an 'Other' category, as their combined frequency is at least 30.


In [29]:
df_hotel['agent'].unique()

array(['0.0', 'Other', '240.0', '15.0', '241.0', '8.0', '250.0', '115.0',
       '5.0', '175.0', '134.0', '156.0', '243.0', '242.0', '3.0', '40.0',
       '147.0', '306.0', '184.0', '96.0', '2.0', '127.0', '95.0', '146.0',
       '9.0', '177.0', '6.0', '143.0', '171.0', '305.0', '67.0', '196.0',
       '152.0', '142.0', '261.0', '104.0', '36.0', '26.0', '29.0', '71.0',
       '181.0', '251.0', '69.0', '248.0', '208.0', '314.0', '281.0',
       '273.0', '253.0', '185.0', '330.0', '326.0', '313.0', '38.0',
       '155.0', '68.0', '308.0', '332.0', '94.0', '339.0', '375.0',
       '66.0', '387.0', '298.0', '91.0', '245.0', '385.0', '168.0',
       '249.0', '315.0', '75.0', '11.0', '436.0', '1.0', '201.0', '183.0',
       '368.0', '464.0', '10.0', '154.0', '468.0', '410.0', '390.0',
       '440.0', '495.0', '493.0', '434.0', '531.0', '16.0', '34.0',
       '47.0', '195.0', '159.0', '78.0', '467.0', '527.0', '479.0',
       '13.0', '7.0', '27.0', '14.0', '22.0', '17.0', '28.0', '42.0',
    

In the categorical columns, values with fewer than 30 occurrences were grouped together as a new "Other" category. We made sure that the combined total of these rare values is at least 30 to maintain meaningful grouping.

In [30]:
df_hotel.shape

(118727, 26)

After the final review of the data cleaning process, we see that 118,727 rows and 26 columns remain in the dataset.

In [31]:
df_hotel.isna().any().any()

np.False_

We confirmed that there are no remaining null values in the dataset, which proves that our data cleaning steps were executed correctly.

In [32]:
df_hotel.sample(5)

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_day_of_month,meal,country,market_segment,distribution_channel,is_repeated_guest,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,city,temperature_2m_mean,rain_sum,people,same_room,total_stays
43441,City Hotel - Lucknow,1,74,February,9,HB,PRT,Offline TA/TO,TA/TO,0,...,Transient-Party,101.50,0,0,Lucknow,13.735417,0.0,2.0,Yes,2
100119,City Hotel - Hyderabad,0,1,April,16,BB,PRT,Direct,Direct,0,...,Transient,129.00,0,1,Hyderabad,31.887503,0.0,2.0,Yes,1
104439,City Hotel - Ahmedabad,0,65,January,16,BB,DEU,Groups,TA/TO,0,...,Transient-Party,65.00,0,0,Ahmedabad,18.706247,0.0,1.0,Yes,2
113355,City Hotel - Ahmedabad,0,434,March,27,BB,AUS,Groups,TA/TO,0,...,Transient-Party,112.67,0,1,Ahmedabad,33.131252,0.0,2.0,Yes,3
3082,Resort Hotel - Bhopal,1,54,June,25,BB,PRT,Online TA,TA/TO,0,...,Transient,62.83,0,1,Bhopal,27.914581,1.7,2.0,Yes,7


In [33]:
df_hotel.to_csv(r"C:\Users\Elvin Aliyev\Downloads\clean_hotel_data.csv", index=False)

Finally, we save the cleaned dataset for use in EDA and machine learning.